# Time discretization
\
CADET uses IDAS from the [<u>SUNDIALS software package</u>](https://sundials.readthedocs.io) to integrate the spatially semi-discretized equations in time.
Importantly, IDAS can handle ODAE and parameter sensitivities.

- relTol and absTol IDAS explanation
- stiffness
- Jacobian to solve the linear system within the Newton iteration for the non-linear system

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive
import ipywidgets as widgets
import os

from cadet import Cadet
import utility.convergence as convergence
import utility.setting_Col1D_SMA_4comp_LWE_benchmark1 as lwe

Cadet.cadet_path = r"C:\Users\jmbr\OneDrive\Desktop\CADET_compiled\master5_generalizedUnit_f1a1972\aRELEASE"
path = os.getcwd()

def graph_column(idas_abstol=1e-1, idas_reltol=1e-1):

    model = Cadet()
    model.root = lwe.get_model(
        spatial_method_bulk=0,
        spatial_method_particle=0,
        particle_type='GENERAL_RATE_PARTICLE',
        axRefinement=2,
        parZ=4,
        return_bulk=True,
        idas_abstol=idas_abstol,
        idas_reltol=idas_reltol
    )
    
    model.filename = 'test.h5'
    model.save()
    model.run_simulation()

    outlet = convergence.get_outlet(path+'/'+model.filename, unit="000")
    sol_time = convergence.get_solution_times(path+'/'+model.filename)
    
    reference = convergence.get_outlet(path+'/data/ref_LWE.h5', unit="000")
    errorComp = np.max(abs(reference[:, 1:] - outlet[:, 1:])) / np.max(abs(reference[:, 1:]))
    errorSalt = np.max(abs(reference[:, 0] - outlet[:, 0])) / np.max(abs(reference[:, 0]))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # First plot
    axs[0].plot(sol_time, outlet[:, 0])
    axs[0].set_xlabel(r'$x~/~M$')
    axs[0].set_ylabel(r'$concentration~/~mol \cdot M^{-3}$')
    axs[0].set_title("Salt")
    
    axs[1].plot(sol_time, outlet[:, 1:])
    axs[1].set_xlabel(r'$x~/~M$')
    axs[1].set_title("components")

    fig.suptitle(
        f"Sim. time: {convergence.get_compute_time(path + '/' + model.filename):.3e}" + f", rel. max. error salt: {errorSalt*100:.2f}%," + f" rel. max. error comps: {errorComp*100:.2f}%",
        fontsize=14
    )
    
    plt.tight_layout()
    plt.show()

abstol_steps = [10**i for i in range(-12, 3)]
abstol_options = [(f"{v:.0e}", v) for v in abstol_steps]

reltol_steps = [10**i for i in range(-12, 3)]
reltol_options = [(f"{v:.0e}", v) for v in reltol_steps]

interact(
    graph_column,
    idas_abstol=widgets.SelectionSlider(
        options=abstol_options,
        description="abstol"
    ),
    idas_reltol=widgets.SelectionSlider(
        options=reltol_options,
        description="reltol"
    )
)



interactive(children=(SelectionSlider(description='abstol', options=(('1e-12', 1e-12), ('1e-11', 1e-11), ('1e-…

<function __main__.graph_column(idas_abstol=0.1, idas_reltol=0.1)>

### Discussion
Error bounds for the local truncation error test are specified by an absolute tolerance (`ABSTOL`) and a relative tolerance (`RELTOL`). Note that the relative tolerance only works for non-zero values, whereas zero values are accounted for by the absolute tolerance. For example, a relative tolerance of $10^{-4}$ and absolute tolerance of $10^{-8}$ requests $3$ significant digits (correct digits after the comma in scientific notation) and considers all numbers with magnitude smaller than $10^{-8}$ as zero.

$|error| < ABSTOL + RELTOL \cdot |y|$

Choosing a reasonable tolerance is very relevant when compute time is limiting, otherwise just set the tolerances low.
Rule of thumb: